# Capybara GitHub + Read the Docs Deployment Broker

This notebook stages the reconstructed repository, creates/opens the GitHub repository, pushes the source with normal Git, reconnects the **existing** Read the Docs project, triggers a build, watches the result, and verifies the public site/offline formats.

**Safety posture:** authentication/MFA remain manual. The notebook does not store passwords, GitHub tokens, Read the Docs tokens, or browser cookies. Consequential actions require an explicit typed confirmation.


In [1]:
from pathlib import Path
import json, os, shutil, subprocess, time, getpass
from urllib.parse import urlparse

REPO_DIR = Path(r".")
GITHUB_OWNER = "BrianBowers-NapaCounty"
GITHUB_REPO = "itam-itsm_integration_framework"
GITHUB_VISIBILITY = "public"
GIT_REMOTE_MODE = "https"   # "https" or "ssh"
RTD_PROJECT_SLUG = "capybara-framework"
RTD_VERSION = "latest"
SELF_HOST_URL = "http://gis.napa.ca.gov/Data/Brian/CCISDA-CSAC/"
SELF_HOST_LOCAL_PATH = None  # Optional local/network path, e.g. r"Z:\Data\Brian\CCISDA-CSAC"

def run(cmd, cwd=REPO_DIR, check=True):
    print('>', ' '.join(map(str,cmd)))
    return subprocess.run(cmd,cwd=cwd,text=True,capture_output=True,check=check)


In [3]:
# Preflight: run before any remote action.
assert REPO_DIR.exists(), REPO_DIR
for required in [".readthedocs.yaml","docs/conf.py","README.md","RELEASE_CHECKLIST.md"]:
    assert (REPO_DIR/required).exists(), required
r=run([os.sys.executable,"tools/release_check.py","--source-only"],check=False)
print(r.stdout); print(r.stderr)
if r.returncode:
    raise RuntimeError("Fix release-check errors before deployment.")
print("Preflight passed.")


> C:\Program Files\ArcGIS\Pro\bin\Python\envs\arcgispro-py3\python.exe tools/release_check.py --source-only
Errors: 0  Warnings: 0


Preflight passed.


In [5]:
# Optional: build local offline artifacts before publishing.
r=run([os.sys.executable,"tools/build_release.py"],check=False)
print(r.stdout); print(r.stderr)
if r.returncode:
    print("Local release build did not complete. This does not modify any remote service.")


> C:\Program Files\ArcGIS\Pro\bin\Python\envs\arcgispro-py3\python.exe tools/build_release.py

Traceback (most recent call last):
  File "C:\Users\BBOWERS\Jupyter Notebooks\Capybara_Framework_GitHub_Repository_RC1\tools\build_release.py", line 32, in <module>
    subprocess.run(base+['-o',str(OUT/'capybara-framework-expanded.docx')],check=True)
  File "C:\Program Files\ArcGIS\Pro\bin\Python\envs\arcgispro-py3\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Program Files\ArcGIS\Pro\bin\Python\envs\arcgispro-py3\Lib\subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "C:\Program Files\ArcGIS\Pro\bin\Python\envs\arcgispro-py3\Lib\subprocess.py", line 1538, in _execute_child
    hp, ht, pid, tid = _winapi.CreateProcess(executable, args,
                       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
FileNotFoundError: [WinError 2] The system cannot

In [6]:
# Selenium browser. Log in manually when prompted by GitHub/RTD/MFA.
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

opts=webdriver.ChromeOptions()
opts.add_experimental_option("detach", True)
driver=webdriver.Chrome(options=opts)
wait=WebDriverWait(driver,30)

def open_github_login():
    driver.get("https://github.com/login")
    print("Sign in manually, complete MFA if requested, then return to the notebook.")

def open_rtd_login():
    driver.get("https://app.readthedocs.org/accounts/login/")
    print("Sign in manually, complete any provider/MFA flow, then return to the notebook.")


In [7]:
def github_repo_url():
    return f"https://github.com/{GITHUB_OWNER}/{GITHUB_REPO}"

def open_or_create_github_repo():
    driver.get(github_repo_url())
    time.sleep(2)
    if "Page not found" not in driver.page_source and driver.current_url.rstrip('/') == github_repo_url().rstrip('/'):
        print("Repository already exists:", github_repo_url()); return
    driver.get("https://github.com/new")
    print("GitHub's repository-creation page is open.")
    print("Create an EMPTY repository named",GITHUB_REPO,"under",GITHUB_OWNER,
          "(do not initialize README/.gitignore/license because the payload already contains them).")
    print("The notebook deliberately leaves the final Create Repository click to you so owner/visibility are visibly confirmed.")


In [8]:
def push_repository():
    confirmation=input(f"Type PUSH {GITHUB_OWNER}/{GITHUB_REPO} to initialize/commit/push this repository: ")
    if confirmation != f"PUSH {GITHUB_OWNER}/{GITHUB_REPO}":
        raise RuntimeError("Push cancelled")
    if not (REPO_DIR/'.git').exists(): run(['git','init'])
    run(['git','add','.'])
    status=run(['git','status','--porcelain'],check=False).stdout.strip()
    if status:
        run(['git','commit','-m','Reconstruct and expand Capybara Framework documentation'])
    run(['git','branch','-M','main'])
    remote = (f"git@github.com:{GITHUB_OWNER}/{GITHUB_REPO}.git" if GIT_REMOTE_MODE=='ssh'
              else f"https://github.com/{GITHUB_OWNER}/{GITHUB_REPO}.git")
    existing=run(['git','remote'],check=False).stdout.split()
    if 'origin' in existing: run(['git','remote','set-url','origin',remote])
    else: run(['git','remote','add','origin',remote])
    print("Git authentication is delegated to your existing SSH key, Git Credential Manager, or GitHub CLI login.")
    result=run(['git','push','-u','origin','main'],check=False)
    print(result.stdout); print(result.stderr)
    if result.returncode: raise RuntimeError("git push failed; authenticate with GitHub and rerun this cell")
    print("Pushed:",github_repo_url())


In [9]:
# Optional GitHub App authorization helper. RTD recommends its GitHub App for an existing-project repository connection.
def open_rtd_github_connection_pages():
    driver.get(f"https://app.readthedocs.org/dashboard/{RTD_PROJECT_SLUG}/edit/")
    print("In Read the Docs Project Settings, use Connected repository to select the new GitHub repository.")
    print("If it is not listed, install/grant the Read the Docs GitHub App access to that repository, then refresh this page.")


In [10]:
# Same-origin Read the Docs API helper using the authenticated Selenium session.
def rtd_fetch(path, method="GET", body=None):
    script=r"""
    const done=arguments[arguments.length-1];
    const path=arguments[0], method=arguments[1], body=arguments[2];
    function cookie(name){return document.cookie.split('; ').find(x=>x.startsWith(name+'='))?.split('=').slice(1).join('=') || '';}
    const headers={'Accept':'application/json'};
    if(method!=='GET' && method!=='HEAD'){
      headers['Content-Type']='application/json';
      const csrf=decodeURIComponent(cookie('csrftoken'));
      if(csrf) headers['X-CSRFToken']=csrf;
    }
    fetch(path,{method,credentials:'include',headers,body:body?JSON.stringify(body):undefined})
      .then(async r=>done({status:r.status,text:await r.text(),url:r.url}))
      .catch(e=>done({status:0,text:String(e)}));
    """
    result=driver.execute_async_script(script,path,method,body)
    text=result.get('text','')
    try: data=json.loads(text) if text else None
    except: data=text
    if result['status'] >= 400: raise RuntimeError(result)
    return result['status'],data

# Ensure the browser is on app.readthedocs.org and authenticated before using session API calls.
driver.get(f"https://app.readthedocs.org/projects/{RTD_PROJECT_SLUG}/")


In [11]:
def inspect_rtd_project():
    status,data=rtd_fetch(f"/api/v3/projects/{RTD_PROJECT_SLUG}/")
    print(json.dumps(data,indent=2))
    return data

project=inspect_rtd_project()


{
  "_links": {
    "_self": "https://app.readthedocs.org/api/v3/projects/capybara-framework/",
    "builds": "https://app.readthedocs.org/api/v3/projects/capybara-framework/builds/",
    "environmentvariables": "https://app.readthedocs.org/api/v3/projects/capybara-framework/environmentvariables/",
    "notifications": "https://app.readthedocs.org/api/v3/projects/capybara-framework/notifications/",
    "redirects": "https://app.readthedocs.org/api/v3/projects/capybara-framework/redirects/",
    "subprojects": "https://app.readthedocs.org/api/v3/projects/capybara-framework/subprojects/",
    "superproject": "https://app.readthedocs.org/api/v3/projects/capybara-framework/superproject/",
    "sync_versions": "https://app.readthedocs.org/api/v3/projects/capybara-framework/sync-versions/",
    "translations": "https://app.readthedocs.org/api/v3/projects/capybara-framework/translations/",
    "versions": "https://app.readthedocs.org/api/v3/projects/capybara-framework/versions/"
  },
  "creat

In [17]:
def reconnect_rtd_project():
    new_repo=github_repo_url()
    confirm=input(f"Type RECONNECT {RTD_PROJECT_SLUG} to point the EXISTING RTD project at {new_repo}: ")
    if confirm != f"RECONNECT {RTD_PROJECT_SLUG}": raise RuntimeError("Reconnect cancelled")
    body={"repository":{"url":new_repo,"type":"git"},"default_branch":"main","default_version":RTD_VERSION}
    status,data=rtd_fetch(f"/api/v3/projects/{RTD_PROJECT_SLUG}/","PATCH",body)
    print("PATCH status",status,data)
    print("Repository setting updated. Connect/select the GitHub repository in the RTD Settings UI as well so push webhooks/status integration are enabled.")


In [18]:
def sync_rtd_versions():
    status,data=rtd_fetch(f"/api/v3/projects/{RTD_PROJECT_SLUG}/sync-versions/","POST",{})
    print("Version sync:",status,data)

def trigger_rtd_build(version=RTD_VERSION):
    confirm=input(f"Type BUILD {version} to trigger a Read the Docs build: ")
    if confirm != f"BUILD {version}": raise RuntimeError("Build cancelled")
    status,data=rtd_fetch(f"/api/v3/projects/{RTD_PROJECT_SLUG}/versions/{version}/builds/","POST",{})
    print("Build trigger:",status,data)
    return data


In [19]:
def recent_builds(limit=10):
    _,data=rtd_fetch(f"/api/v3/projects/{RTD_PROJECT_SLUG}/builds/?limit={limit}")
    return data

def watch_build(timeout_minutes=20):
    deadline=time.time()+timeout_minutes*60
    last=None
    while time.time()<deadline:
        data=recent_builds(5)
        build=data['results'][0] if data.get('results') else None
        if build!=last:
            print(json.dumps(build,indent=2)); last=build
        if build and not build.get('running',False) and build.get('state') in {'finished','cancelled'}:
            return build
        time.sleep(10)
    raise TimeoutError("Build did not finish within the requested watch window")


In [22]:
import getpass
import requests

RTD_API_TOKEN = getpass.getpass(
    "Read the Docs API token (hidden; not saved): "
).strip()

def rtd_fetch(path, method="GET", body=None):
    url = (
        path if path.startswith("http")
        else "https://app.readthedocs.org/api/v3/" + path.lstrip("/")
    )

    headers = {
        "Accept": "application/json",
        "Authorization": f"Token {RTD_API_TOKEN}",
    }

    if method.upper() not in {"GET", "HEAD"}:
        headers["Content-Type"] = "application/json"

    response = requests.request(
        method.upper(),
        url,
        headers=headers,
        json=body if body is not None else None,
        timeout=60,
    )

    try:
        data = response.json() if response.text else None
    except Exception:
        data = response.text

    if response.status_code >= 400:
        raise RuntimeError({
            "status": response.status_code,
            "data": data,
            "url": response.url,
        })

    return response.status_code, data
    


Read the Docs API token (hidden; not saved):  ········


In [23]:
project = inspect_rtd_project()
sync_rtd_versions()
build = trigger_rtd_build()

RuntimeError: {'status': 404, 'data': '\n\n\n<!DOCTYPE html>\n<html lang="en" data-theme="">\n  <head>\n\n    \n    \n    <title>\n  Page not found\n - Read the Docs Community </title>\n    \n\n    \n      <meta name="description"\n            content="Read the Docs is a documentation publishing and hosting platform for technical documentation" />\n      <meta name="keywords" content="documentation hosting" />\n    \n\n    \n      <link rel="icon" type="image/png" href="https://app-assets.readthedocs.org/images/favicon.f231b6609d0b.png" />\n      \n      <link rel="stylesheet" type="text/css" href="https://app-assets.readthedocs.org/readthedocsext/theme/css/site.115425a40ba0.css" media="all" />\n      \n        <link rel="stylesheet" type="text/css" href="https://app-assets.readthedocs.org/readthedocsext/theme/css/dark.c8c0feaf47c2.css" media="(prefers-color-scheme: dark)" />\n      \n    \n    \n    \n    \n\n    <meta http-equiv="Content-Type" content="text/html; charset=utf-8" />\n 

In [20]:
# Final live-build gate. Run only after the new repository is connected and the pushed commit is visible.
sync_rtd_versions()
build=trigger_rtd_build()
# finished=watch_build()  # Uncomment/run after triggering if you want the notebook to wait and print state changes.


RuntimeError: {'status': 403, 'text': '{"detail":"CSRF Failed: CSRF token missing."}', 'url': 'https://app.readthedocs.org/api/v3/projects/capybara-framework/sync-versions/'}

In [ ]:
# Verification and offline-format URLs.
import requests
PUBLIC=f"https://{RTD_PROJECT_SLUG}.readthedocs.io/en/{RTD_VERSION}/"
urls={
 "docs":PUBLIC,
 "pdf":f"https://{RTD_PROJECT_SLUG}.readthedocs.io/_/downloads/en/{RTD_VERSION}/pdf/",
 "epub":f"https://{RTD_PROJECT_SLUG}.readthedocs.io/_/downloads/en/{RTD_VERSION}/epub/",
 "htmlzip":f"https://{RTD_PROJECT_SLUG}.readthedocs.io/_/downloads/en/{RTD_VERSION}/htmlzip/",
}
for name,url in urls.items():
    try:
        r=requests.get(url,timeout=30,allow_redirects=True,stream=True)
        print(name,r.status_code,r.url,r.headers.get('content-type'))
        r.close()
    except Exception as e: print(name,'ERROR',e)


In [ ]:
# Optional self-hosting copy for DOCX + embedded single HTML.
def copy_self_hosted_exports():
    release=REPO_DIR/'release'
    files=[release/'capybara-framework-expanded.html',release/'capybara-framework-expanded.docx']
    for f in files: assert f.exists(),f
    if SELF_HOST_LOCAL_PATH:
        dest=Path(SELF_HOST_LOCAL_PATH);dest.mkdir(parents=True,exist_ok=True)
        for f in files:
            shutil.copy2(f,dest/f.name);print('Copied',f.name,'->',dest)
    else:
        print('Upload these files to the web directory serving',SELF_HOST_URL)
        for f in files:print(' -',f)
        print('Expected URLs:')
        for f in files:print(SELF_HOST_URL.rstrip('/')+'/'+f.name)
copy_self_hosted_exports()


In [ ]:
# Optional: create a GitHub release with offline artifacts if GitHub CLI is installed/authenticated.
def publish_release_assets(tag="v0.2.0-rc1"):
    gh=shutil.which('gh')
    if not gh:
        print('GitHub CLI not found; skip release-asset upload or install gh.');return
    confirm=input(f"Type RELEASE {tag} to create/upload GitHub release assets: ")
    if confirm != f"RELEASE {tag}": raise RuntimeError('Release cancelled')
    assets=[str(p) for p in (REPO_DIR/'release').iterdir() if p.is_file()]
    cmd=[gh,'release','create',tag,*assets,'--title',f'Capybara Framework {tag}','--notes','Reconstructed and expanded release candidate.']
    r=subprocess.run(cmd,cwd=REPO_DIR,text=True,capture_output=True)
    print(r.stdout);print(r.stderr)


## Recommended deployment order

1. Run preflight and local release build.
2. Sign into GitHub manually; create/open an empty public repository.
3. Push the repository with Git.
4. Grant the Read the Docs GitHub App access and select the repository in the existing RTD project's **Connected repository** setting.
5. Inspect the RTD API project record.
6. Reconnect the existing project to the new GitHub URL and `main` branch.
7. Sync versions.
8. Trigger a `latest` build and inspect the build result/logs.
9. Verify the public documentation and RTD PDF/ePub/HTML.ZIP URLs.
10. Copy the self-contained HTML and DOCX to the maintainer server if desired.
11. Optionally publish the offline files as GitHub Release assets.

Do not delete/recreate the Read the Docs project; preserving the existing project preserves its public identity and inbound links.


In [24]:
# =====================================================================
# CAPYBARA FRAMEWORK — COMPLETE READ THE DOCS DEPLOYMENT / GO-LIVE CELL
# =====================================================================
#
# Prerequisites:
#   1. Your Selenium driver is already open and stored in variable: driver
#   2. The new GitHub repository already contains the Capybara repository
#   3. You have a Read the Docs API token
#
# This cell will:
#   - verify the open Selenium browser
#   - authenticate to RTD API using your token
#   - inspect the existing capybara-framework project
#   - verify/update its GitHub repository URL
#   - verify/update default branch and .readthedocs.yaml path
#   - optionally make "latest" the default version
#   - sync RTD versions
#   - activate/unhide "latest" if necessary
#   - trigger a fresh latest build
#   - monitor the build until it finishes
#   - verify PDF / EPUB / HTML.ZIP download URLs
#   - open the live documentation in Selenium
#
# The token is NEVER written to disk.
# =====================================================================

import requests
import getpass
import time
import json
import re
from urllib.parse import urlparse

# ---------------------------------------------------------------------
# SETTINGS
# ---------------------------------------------------------------------

RTD_PROJECT_SLUG = globals().get("RTD_PROJECT_SLUG", "capybara-framework")
RTD_VERSION      = globals().get("RTD_VERSION", "latest")

DEFAULT_BRANCH = "main"

# Our repo uses the root-level .readthedocs.yaml
RTD_YAML_PATH = ".readthedocs.yaml"

# Set True if the root RTD URL should resolve to "latest".
MAKE_LATEST_DEFAULT = True

API_BASE = "https://app.readthedocs.org/api/v3"

# Try to inherit the GitHub information from earlier notebook cells.
GITHUB_OWNER = globals().get("GITHUB_OWNER", "").strip()
GITHUB_REPO  = globals().get("GITHUB_REPO", "capybara-framework").strip()

EXPECTED_REPO = (
    globals().get("EXPECTED_GITHUB_REPO_URL", "")
    or globals().get("GITHUB_REPO_URL", "")
).strip()

if not EXPECTED_REPO and GITHUB_OWNER:
    EXPECTED_REPO = f"https://github.com/{GITHUB_OWNER}/{GITHUB_REPO}"

# ---------------------------------------------------------------------
# HELPERS
# ---------------------------------------------------------------------

def normalize_repo_url(url):
    if not url:
        return ""
    url = url.strip().rstrip("/")
    if url.endswith(".git"):
        url = url[:-4]
    return url.lower()


def pretty(value):
    print(json.dumps(value, indent=2, default=str))


def api(method, path, body=None, allowed=(200, 201, 202, 204)):
    """Token-authenticated Read the Docs API call."""
    url = path if path.startswith("http") else API_BASE + "/" + path.lstrip("/")

    headers = {
        "Accept": "application/json",
        "Authorization": f"Token {RTD_API_TOKEN}",
        "User-Agent": "Capybara-Framework-Deployment-Broker/1.0",
    }

    if body is not None:
        headers["Content-Type"] = "application/json"

    r = requests.request(
        method.upper(),
        url,
        headers=headers,
        json=body,
        timeout=60,
    )

    if r.text:
        try:
            data = r.json()
        except Exception:
            data = r.text
    else:
        data = None

    if r.status_code not in allowed:
        raise RuntimeError({
            "method": method,
            "status": r.status_code,
            "url": url,
            "response": data,
        })

    return r.status_code, data


def build_state(build):
    state = build.get("state")
    if isinstance(state, dict):
        return state.get("code") or state.get("name") or str(state)
    return str(state or "")


def extract_build_id(payload):
    """Handle several RTD response shapes."""
    if not payload:
        return None

    build = payload.get("build") if isinstance(payload, dict) else None

    if isinstance(build, dict):
        if build.get("id"):
            return int(build["id"])

        link = (
            build.get("_links", {}).get("_self")
            or build.get("url")
        )
        if link:
            m = re.search(r"/builds/(\d+)/?", link)
            if m:
                return int(m.group(1))

    if isinstance(build, int):
        return build

    if isinstance(build, str):
        if build.isdigit():
            return int(build)
        m = re.search(r"/builds/(\d+)/?", build)
        if m:
            return int(m.group(1))

    return None


def list_builds(limit=20):
    _, data = api(
        "GET",
        f"/projects/{RTD_PROJECT_SLUG}/builds/?limit={limit}"
    )
    return data.get("results", [])


def find_new_build(old_ids, version):
    """Wait briefly for a newly-triggered RTD build to appear."""
    for _ in range(30):
        builds = list_builds(30)

        for b in builds:
            bid = b.get("id")
            bversion = b.get("version")

            if isinstance(bversion, dict):
                bversion = bversion.get("slug")

            if bid not in old_ids and bversion == version:
                return bid

        time.sleep(2)

    return None


# ---------------------------------------------------------------------
# 1. VERIFY THE EXISTING SELENIUM CONNECTION
# ---------------------------------------------------------------------

print("=" * 72)
print("CAPYBARA FRAMEWORK — READ THE DOCS GO-LIVE")
print("=" * 72)

try:
    current_title = driver.title
    print("✓ Selenium session is alive")
    print("  Browser title:", current_title)
except Exception as exc:
    raise RuntimeError(
        "The Selenium driver is not available anymore. "
        "Restart/reconnect Selenium before running this cell."
    ) from exc


# ---------------------------------------------------------------------
# 2. LOAD RTD API TOKEN
# ---------------------------------------------------------------------

RTD_API_TOKEN = globals().get("RTD_API_TOKEN", "").strip()

if not RTD_API_TOKEN:
    RTD_API_TOKEN = getpass.getpass(
        "Read the Docs API token "
        "(hidden; retained only in kernel memory): "
    ).strip()

if not RTD_API_TOKEN:
    raise RuntimeError("No Read the Docs API token supplied.")

globals()["RTD_API_TOKEN"] = RTD_API_TOKEN


# ---------------------------------------------------------------------
# 3. VERIFY TOKEN + PROJECT
# ---------------------------------------------------------------------

print("\n[1/9] Inspecting existing Read the Docs project...")

_, project = api(
    "GET",
    f"/projects/{RTD_PROJECT_SLUG}/"
)

if project.get("slug") != RTD_PROJECT_SLUG:
    raise RuntimeError(
        f"Unexpected RTD project returned: {project.get('slug')!r}"
    )

print("✓ Project:", project.get("name"))
print("✓ Slug:   ", project.get("slug"))

current_repo = (
    project.get("repository", {}).get("url")
    if isinstance(project.get("repository"), dict)
    else ""
)

print("✓ Current repository:", current_repo)
print("✓ Default branch:    ", project.get("default_branch"))
print("✓ Default version:   ", project.get("default_version"))


# ---------------------------------------------------------------------
# 4. DETERMINE / VERIFY EXPECTED GITHUB REPOSITORY
# ---------------------------------------------------------------------

if not EXPECTED_REPO:
    print(
        "\nI could not determine the intended new GitHub repository "
        "from existing notebook variables."
    )
    EXPECTED_REPO = input(
        "Enter the FULL new GitHub repository URL\n"
        "(example: https://github.com/YourName/capybara-framework): "
    ).strip()

if not EXPECTED_REPO.startswith("https://github.com/"):
    raise RuntimeError(
        f"Expected a GitHub HTTPS repository URL, got: {EXPECTED_REPO}"
    )

print("\n[2/9] Verifying GitHub repository...")
print("Expected repository:", EXPECTED_REPO)

# Public GitHub reachability check.
gh = requests.get(EXPECTED_REPO, timeout=30, allow_redirects=True)

if gh.status_code >= 400:
    raise RuntimeError(
        f"GitHub repository is not publicly reachable "
        f"(HTTP {gh.status_code}): {EXPECTED_REPO}"
    )

print("✓ GitHub repository is reachable.")


# ---------------------------------------------------------------------
# 5. DISPLAY DEPLOYMENT PLAN AND REQUIRE ONE FINAL CONFIRMATION
# ---------------------------------------------------------------------

needs_repo_change = (
    normalize_repo_url(current_repo) != normalize_repo_url(EXPECTED_REPO)
)

patch = {}

if needs_repo_change:
    patch["repository"] = {
        "url": EXPECTED_REPO.rstrip("/"),
        "type": "git",
    }

if project.get("default_branch") != DEFAULT_BRANCH:
    patch["default_branch"] = DEFAULT_BRANCH

if project.get("readthedocs_yaml_path") != RTD_YAML_PATH:
    patch["readthedocs_yaml_path"] = RTD_YAML_PATH

if MAKE_LATEST_DEFAULT and project.get("default_version") != RTD_VERSION:
    patch["default_version"] = RTD_VERSION

print("\nDeployment plan")
print("----------------")
print("RTD project:        ", RTD_PROJECT_SLUG)
print("GitHub repository:  ", EXPECTED_REPO)
print("Branch:             ", DEFAULT_BRANCH)
print("RTD config:         ", RTD_YAML_PATH)
print("Version to build:   ", RTD_VERSION)
print("Make latest default:", MAKE_LATEST_DEFAULT)

if patch:
    print("\nProject settings requiring update:")
    pretty(patch)
else:
    print("\n✓ Project settings already match the deployment target.")

confirmation = input(
    f"\nType DEPLOY {RTD_PROJECT_SLUG} to continue: "
).strip()

if confirmation != f"DEPLOY {RTD_PROJECT_SLUG}":
    raise RuntimeError("Deployment cancelled.")


# ---------------------------------------------------------------------
# 6. UPDATE RTD PROJECT SETTINGS IF NEEDED
# ---------------------------------------------------------------------

print("\n[3/9] Verifying Read the Docs project configuration...")

if patch:
    status, _ = api(
        "PATCH",
        f"/projects/{RTD_PROJECT_SLUG}/",
        patch,
        allowed=(200, 204),
    )

    print("✓ Project settings updated:", status)

    _, project = api(
        "GET",
        f"/projects/{RTD_PROJECT_SLUG}/"
    )

else:
    print("✓ No project update required.")

updated_repo = project.get("repository", {}).get("url", "")

if normalize_repo_url(updated_repo) != normalize_repo_url(EXPECTED_REPO):
    raise RuntimeError(
        "RTD repository verification failed.\n"
        f"Expected: {EXPECTED_REPO}\n"
        f"Found:    {updated_repo}"
    )

print("✓ RTD now points to:", updated_repo)


# ---------------------------------------------------------------------
# 7. SYNCHRONIZE VERSIONS
# ---------------------------------------------------------------------

print("\n[4/9] Synchronizing repository versions...")

status, sync_result = api(
    "POST",
    f"/projects/{RTD_PROJECT_SLUG}/sync-versions/",
    None,
    allowed=(202,),
)

print("✓ Version synchronization accepted:", status)


# ---------------------------------------------------------------------
# 8. WAIT FOR "latest" VERSION TO BE AVAILABLE
# ---------------------------------------------------------------------

print(f"\n[5/9] Waiting for RTD version '{RTD_VERSION}'...")

version = None

for attempt in range(45):
    try:
        _, version = api(
            "GET",
            f"/projects/{RTD_PROJECT_SLUG}/versions/{RTD_VERSION}/"
        )
        break
    except RuntimeError as exc:
        if "404" not in str(exc):
            raise
        time.sleep(2)

if version is None:
    raise RuntimeError(
        f"RTD did not expose version '{RTD_VERSION}' after synchronization."
    )

print("✓ Version found")
print("  Version:   ", version.get("slug"))
print("  Identifier:", version.get("identifier"))
print("  Active:    ", version.get("active"))
print("  Built:     ", version.get("built"))


# ---------------------------------------------------------------------
# 9. CAPTURE EXISTING BUILDS BEFORE WE TRIGGER A NEW ONE
# ---------------------------------------------------------------------

old_builds = list_builds(30)
old_build_ids = {
    b.get("id")
    for b in old_builds
    if b.get("id") is not None
}


# ---------------------------------------------------------------------
# 10. ACTIVATE / UNHIDE LATEST IF REQUIRED
# ---------------------------------------------------------------------

activated_now = False

if not version.get("active") or version.get("hidden"):
    print("\n[6/9] Activating/unhiding version...")

    status, _ = api(
        "PATCH",
        f"/projects/{RTD_PROJECT_SLUG}/versions/{RTD_VERSION}/",
        {
            "active": True,
            "hidden": False,
        },
        allowed=(200, 204),
    )

    print("✓ Version activated.")
    activated_now = True

else:
    print("\n[6/9] ✓ Version already active and visible.")


# ---------------------------------------------------------------------
# 11. TRIGGER THE BUILD
# ---------------------------------------------------------------------

print("\n[7/9] Triggering a fresh documentation build...")

# Activating an inactive RTD version may itself trigger a build.
# Give it a few seconds to appear before explicitly triggering another.
build_id = None

if activated_now:
    build_id = find_new_build(old_build_ids, RTD_VERSION)

if build_id:
    print("✓ Activation automatically triggered build:", build_id)

else:
    status, trigger = api(
        "POST",
        f"/projects/{RTD_PROJECT_SLUG}/versions/{RTD_VERSION}/builds/",
        None,
        allowed=(202,),
    )

    print("✓ Build request accepted:", status)

    build_id = extract_build_id(trigger)

    if build_id is None:
        build_id = find_new_build(old_build_ids, RTD_VERSION)

if build_id is None:
    raise RuntimeError(
        "The build request was accepted, but the new build ID "
        "could not be identified."
    )

print("✓ Build ID:", build_id)

build_dashboard_url = (
    f"https://app.readthedocs.org/projects/"
    f"{RTD_PROJECT_SLUG}/builds/{build_id}/"
)

print("  Dashboard:", build_dashboard_url)


# ---------------------------------------------------------------------
# 12. WATCH BUILD TO COMPLETION
# ---------------------------------------------------------------------

print("\n[8/9] Monitoring Read the Docs build...")

last_state = None
build = None

for _ in range(180):

    _, build = api(
        "GET",
        f"/projects/{RTD_PROJECT_SLUG}/builds/{build_id}/?expand=config"
    )

    state = build_state(build).lower()

    if state != last_state:
        print(
            f"  {time.strftime('%H:%M:%S')}  "
            f"state={state}"
        )
        last_state = state

    if state in {"finished", "cancelled"}:
        break

    time.sleep(5)

else:
    driver.get(build_dashboard_url)
    raise RuntimeError(
        "Build monitoring timed out. "
        "The build dashboard has been opened in Selenium."
    )


# ---------------------------------------------------------------------
# 13. HANDLE SUCCESS / FAILURE
# ---------------------------------------------------------------------

success = bool(build.get("success"))

if not success:
    print("\n✗ READ THE DOCS BUILD FAILED")
    print("Build ID:", build_id)
    print("State:   ", build_state(build))
    print("Commit:  ", build.get("commit"))
    print("Error:   ", build.get("error"))

    driver.get(build_dashboard_url)

    raise RuntimeError(
        "Read the Docs build failed. "
        "The build log is now open in Selenium."
    )

print("\n✓ READ THE DOCS BUILD SUCCEEDED")
print("  Build ID:", build_id)
print("  Commit:  ", build.get("commit"))
print("  Duration:", build.get("duration"), "seconds")


# ---------------------------------------------------------------------
# 14. VERIFY PUBLISHED VERSION + DOWNLOADS
# ---------------------------------------------------------------------

print("\n[9/9] Verifying published documentation and offline formats...")

_, final_version = api(
    "GET",
    f"/projects/{RTD_PROJECT_SLUG}/versions/{RTD_VERSION}/"
)

docs_url = (
    final_version.get("urls", {}).get("documentation")
    or project.get("urls", {}).get("documentation")
    or f"https://{RTD_PROJECT_SLUG}.readthedocs.io/en/{RTD_VERSION}/"
)

downloads = final_version.get("downloads") or {}

print("\nLIVE DOCUMENTATION")
print("------------------")
print(docs_url)

print("\nOFFLINE DOWNLOADS")
print("-----------------")

for fmt in ("pdf", "epub", "htmlzip"):
    url = downloads.get(fmt)

    if not url:
        print(f"{fmt.upper():8} : not reported by RTD")
        continue

    try:
        check = requests.get(
            url,
            timeout=30,
            allow_redirects=True,
            stream=True,
        )
        okay = check.status_code < 400
        check.close()
    except Exception:
        okay = False

    mark = "✓" if okay else "?"
    print(f"{mark} {fmt.upper():8} : {url}")


# ---------------------------------------------------------------------
# 15. FINAL PROJECT VERIFICATION
# ---------------------------------------------------------------------

_, final_project = api(
    "GET",
    f"/projects/{RTD_PROJECT_SLUG}/"
)

print("\nFINAL RTD CONFIGURATION")
print("-----------------------")
print(
    "Repository:     ",
    final_project.get("repository", {}).get("url")
)
print(
    "Default branch: ",
    final_project.get("default_branch")
)
print(
    "Default version:",
    final_project.get("default_version")
)
print(
    "Config path:    ",
    final_project.get("readthedocs_yaml_path")
)


# ---------------------------------------------------------------------
# 16. OPEN THE LIVE DOCUMENTATION IN THE EXISTING SELENIUM WINDOW
# ---------------------------------------------------------------------

driver.get(docs_url)

print("\n" + "=" * 72)
print("CAPYBARA FRAMEWORK IS LIVE")
print("=" * 72)
print(docs_url)
print()
print("The live documentation has been opened in your Selenium browser.")
print("RTD build:", build_id)
print("Commit:", build.get("commit"))
print("=" * 72)

CAPYBARA FRAMEWORK — READ THE DOCS GO-LIVE
✓ Selenium session is alive
  Browser title: Edit Project - Read the Docs Community

[1/9] Inspecting existing Read the Docs project...
✓ Project: Capybara Framework
✓ Slug:    capybara-framework
✓ Current repository: https://github.com/brianjbowers/Capybara.git
✓ Default branch:     main
✓ Default version:    latest

[2/9] Verifying GitHub repository...
Expected repository: https://github.com/BrianBowers-NapaCounty/itam-itsm_integration_framework
✓ GitHub repository is reachable.

Deployment plan
----------------
RTD project:         capybara-framework
GitHub repository:   https://github.com/BrianBowers-NapaCounty/itam-itsm_integration_framework
Branch:              main
RTD config:          .readthedocs.yaml
Version to build:    latest
Make latest default: True

Project settings requiring update:
{
  "repository": {
    "url": "https://github.com/BrianBowers-NapaCounty/itam-itsm_integration_framework",
    "type": "git"
  },
  "readthedocs_y


Type DEPLOY capybara-framework to continue:  DEPLOY capybara-framework



[3/9] Verifying Read the Docs project configuration...
✓ Project settings updated: 204
✓ RTD now points to: https://github.com/BrianBowers-NapaCounty/itam-itsm_integration_framework

[4/9] Synchronizing repository versions...
✓ Version synchronization accepted: 202

[5/9] Waiting for RTD version 'latest'...
✓ Version found
  Version:    latest
  Identifier: main
  Active:     True
  Built:      True

[6/9] ✓ Version already active and visible.

[7/9] Triggering a fresh documentation build...
✓ Build request accepted: 202
✓ Build ID: 34538275
  Dashboard: https://app.readthedocs.org/projects/capybara-framework/builds/34538275/

[8/9] Monitoring Read the Docs build...
  13:30:23  state=cloning
  13:30:28  state=finished

✗ READ THE DOCS BUILD FAILED
Build ID: 34538275
State:    finished
Commit:   None
Error:    


RuntimeError: Read the Docs build failed. The build log is now open in Selenium.

In [25]:
# =====================================================================
# CAPYBARA — REPAIR MISSING "main" BRANCH AND REBUILD READ THE DOCS
# =====================================================================

import subprocess
import requests
import time
import json
import re
from pathlib import Path

RTD_PROJECT_SLUG = globals().get("RTD_PROJECT_SLUG", "capybara-framework")
RTD_VERSION = globals().get("RTD_VERSION", "latest")

# ---------------------------------------------------------------------
# 1. FIND THE LOCAL GIT REPOSITORY
# ---------------------------------------------------------------------

possible_roots = [
    Path.cwd(),
    Path.cwd().parent,
    Path.cwd() / "repository",
]

REPO_ROOT = None

for candidate in possible_roots:
    try:
        result = subprocess.run(
            ["git", "-C", str(candidate), "rev-parse", "--show-toplevel"],
            capture_output=True,
            text=True,
        )
        if result.returncode == 0:
            REPO_ROOT = Path(result.stdout.strip())
            break
    except Exception:
        pass

if REPO_ROOT is None:
    raise RuntimeError(
        "Could not locate the local Git repository from the current notebook "
        "directory. Set REPO_ROOT manually to the extracted repository folder."
    )

print("Local repository:")
print(" ", REPO_ROOT)


# ---------------------------------------------------------------------
# 2. INSPECT ORIGIN
# ---------------------------------------------------------------------

def git(*args, check=True):
    result = subprocess.run(
        ["git", "-C", str(REPO_ROOT), *args],
        capture_output=True,
        text=True,
    )

    if check and result.returncode != 0:
        raise RuntimeError(
            f"Git command failed:\n"
            f"git {' '.join(args)}\n\n"
            f"STDOUT:\n{result.stdout}\n"
            f"STDERR:\n{result.stderr}"
        )

    return result


origin = git("remote", "get-url", "origin").stdout.strip()

print("\nGitHub origin:")
print(" ", origin)


# ---------------------------------------------------------------------
# 3. SHOW CURRENT LOCAL + REMOTE BRANCHES
# ---------------------------------------------------------------------

current_branch = git(
    "branch",
    "--show-current"
).stdout.strip()

print("\nCurrent local branch:")
print(" ", current_branch or "(detached HEAD)")

remote_check = subprocess.run(
    ["git", "ls-remote", "--heads", origin],
    capture_output=True,
    text=True,
)

if remote_check.returncode != 0:
    raise RuntimeError(
        "Could not inspect GitHub remote branches:\n"
        + remote_check.stderr
    )

remote_branches = []

for line in remote_check.stdout.splitlines():
    m = re.search(r"refs/heads/(.+)$", line)
    if m:
        remote_branches.append(m.group(1))

print("\nBranches currently on GitHub:")
if remote_branches:
    for branch in remote_branches:
        print("  -", branch)
else:
    print("  (none)")


# ---------------------------------------------------------------------
# 4. ENSURE THE LOCAL REPOSITORY HAS A COMMIT
# ---------------------------------------------------------------------

commit_check = git(
    "rev-parse",
    "--verify",
    "HEAD",
    check=False,
)

if commit_check.returncode != 0:
    raise RuntimeError(
        "The local repository does not contain a commit yet. "
        "The GitHub repository therefore cannot have a buildable branch."
    )

commit = commit_check.stdout.strip()

print("\nLocal HEAD commit:")
print(" ", commit)


# ---------------------------------------------------------------------
# 5. RENAME CURRENT LOCAL BRANCH TO main
# ---------------------------------------------------------------------

if current_branch != "main":

    print(
        f"\nRenaming local branch "
        f"{current_branch!r} -> 'main'..."
    )

    git("branch", "-M", "main")

else:
    print("\n✓ Local branch is already 'main'.")


# ---------------------------------------------------------------------
# 6. PUSH main TO GITHUB
# ---------------------------------------------------------------------

print("\nPushing main to GitHub...")

push = subprocess.run(
    [
        "git",
        "-C",
        str(REPO_ROOT),
        "push",
        "-u",
        "origin",
        "main",
    ],
    text=True,
)

if push.returncode != 0:
    raise RuntimeError(
        "\nGit could not push 'main'.\n\n"
        "If GitHub asks you to authenticate, complete that authentication "
        "using your normal Git Credential Manager / browser login and then "
        "run this cell again."
    )

print("✓ Push completed.")


# ---------------------------------------------------------------------
# 7. VERIFY main REALLY EXISTS REMOTELY
# ---------------------------------------------------------------------

verify = subprocess.run(
    [
        "git",
        "ls-remote",
        "--heads",
        origin,
        "refs/heads/main",
    ],
    capture_output=True,
    text=True,
)

if verify.returncode != 0 or not verify.stdout.strip():
    raise RuntimeError(
        "Git push appeared to finish, but refs/heads/main still "
        "cannot be found on GitHub."
    )

remote_main_commit = verify.stdout.split()[0]

print("\n✓ GitHub now has:")
print("  refs/heads/main")
print("  commit:", remote_main_commit)


# ---------------------------------------------------------------------
# 8. VERIFY / LOAD RTD TOKEN
# ---------------------------------------------------------------------

if not globals().get("RTD_API_TOKEN"):
    import getpass

    RTD_API_TOKEN = getpass.getpass(
        "Read the Docs API token (hidden): "
    ).strip()

    globals()["RTD_API_TOKEN"] = RTD_API_TOKEN

else:
    RTD_API_TOKEN = globals()["RTD_API_TOKEN"]


API_BASE = "https://app.readthedocs.org/api/v3"

def rtd(method, path, body=None, allowed=(200, 201, 202, 204)):

    url = API_BASE + "/" + path.lstrip("/")

    headers = {
        "Accept": "application/json",
        "Authorization": f"Token {RTD_API_TOKEN}",
    }

    if body is not None:
        headers["Content-Type"] = "application/json"

    response = requests.request(
        method,
        url,
        headers=headers,
        json=body,
        timeout=60,
    )

    try:
        data = response.json() if response.text else None
    except Exception:
        data = response.text

    if response.status_code not in allowed:
        raise RuntimeError({
            "status": response.status_code,
            "url": url,
            "response": data,
        })

    return response.status_code, data


# ---------------------------------------------------------------------
# 9. FORCE RTD DEFAULT BRANCH TO main
# ---------------------------------------------------------------------

print("\nUpdating Read the Docs default branch...")

status, _ = rtd(
    "PATCH",
    f"projects/{RTD_PROJECT_SLUG}/",
    {
        "default_branch": "main",
    },
)

print("✓ RTD default_branch = main")


# ---------------------------------------------------------------------
# 10. RESYNC RTD VERSIONS
# ---------------------------------------------------------------------

print("\nSynchronizing versions...")

status, result = rtd(
    "POST",
    f"projects/{RTD_PROJECT_SLUG}/sync-versions/",
)

print("✓ Version sync accepted:", status)

# Give the asynchronous sync a moment to settle.
time.sleep(5)


# ---------------------------------------------------------------------
# 11. VERIFY LATEST
# ---------------------------------------------------------------------

status, version = rtd(
    "GET",
    f"projects/{RTD_PROJECT_SLUG}/versions/{RTD_VERSION}/",
)

print("\nRTD version:")
print("  slug:      ", version.get("slug"))
print("  identifier:", version.get("identifier"))
print("  active:    ", version.get("active"))

# If latest still retained the old identifier, update it.
identifier = version.get("identifier")

if identifier != "main":

    print(
        "\nThe 'latest' version is still pointing to "
        f"{identifier!r}."
    )

    print(
        "Synchronizing again now that main exists..."
    )

    rtd(
        "POST",
        f"projects/{RTD_PROJECT_SLUG}/sync-versions/",
    )

    time.sleep(5)

    _, version = rtd(
        "GET",
        f"projects/{RTD_PROJECT_SLUG}/versions/{RTD_VERSION}/",
    )

    print(
        "Latest identifier after sync:",
        version.get("identifier"),
    )


# ---------------------------------------------------------------------
# 12. TRIGGER FRESH BUILD
# ---------------------------------------------------------------------

print("\nTriggering fresh Read the Docs build...")

status, build_result = rtd(
    "POST",
    (
        f"projects/{RTD_PROJECT_SLUG}/"
        f"versions/{RTD_VERSION}/builds/"
    ),
)

print("✓ Build accepted:", status)

print("\nBuild response:")
print(json.dumps(build_result, indent=2, default=str))


# ---------------------------------------------------------------------
# 13. OPEN RTD BUILD PAGE IN EXISTING SELENIUM SESSION
# ---------------------------------------------------------------------

builds_url = (
    f"https://app.readthedocs.org/projects/"
    f"{RTD_PROJECT_SLUG}/builds/"
)

driver.get(builds_url)

print("\n" + "=" * 70)
print("REPAIR COMPLETE")
print("=" * 70)

print("GitHub branch:")
print("  main")

print("\nCommit:")
print(" ", remote_main_commit)

print("\nRead the Docs:")
print("  default_branch = main")

print("\nThe Read the Docs build page is now open in Selenium.")
print("=" * 70)

RuntimeError: Could not locate the local Git repository from the current notebook directory. Set REPO_ROOT manually to the extracted repository folder.

In [26]:
GITHUB_OWNER = "BrianBowers-NapaCounty"
GITHUB_REPO = "itam-itsm_integration_framework"

EXPECTED_REPO = (
    "https://github.com/"
    "BrianBowers-NapaCounty/"
    "itam-itsm_integration_framework"
)

RTD_PROJECT_SLUG = "capybara-framework"
RTD_VERSION = "latest"
DEFAULT_BRANCH = "main"

In [27]:
import subprocess

subprocess.run(
    ["git", "remote", "-v"],
    text=True
)

CompletedProcess(args=['git', 'remote', '-v'], returncode=128)

In [28]:
subprocess.run([
    "git",
    "remote",
    "set-url",
    "origin",
    "https://github.com/BrianBowers-NapaCounty/itam-itsm_integration_framework.git"
], check=True)


CalledProcessError: Command '['git', 'remote', 'set-url', 'origin', 'https://github.com/BrianBowers-NapaCounty/itam-itsm_integration_framework.git']' returned non-zero exit status 128.

In [ ]:
# =====================================================================
# CAPYBARA — REPAIR LOCAL GIT REPOSITORY + ORIGIN + MAIN + PUSH
# =====================================================================

from pathlib import Path
import subprocess
import os

GITHUB_OWNER = "BrianBowers-NapaCounty"
GITHUB_REPO = "itam-itsm_integration_framework"

GITHUB_URL = (
    f"https://github.com/{GITHUB_OWNER}/{GITHUB_REPO}.git"
)

# ---------------------------------------------------------------------
# HELPERS
# ---------------------------------------------------------------------

def run_git(repo, *args, check=True):
    result = subprocess.run(
        ["git", "-C", str(repo), *args],
        capture_output=True,
        text=True,
    )

    if result.stdout.strip():
        print(result.stdout.strip())

    if result.stderr.strip():
        print(result.stderr.strip())

    if check and result.returncode != 0:
        raise RuntimeError(
            f"\nGit command failed ({result.returncode}):\n"
            f"git {' '.join(args)}\n\n"
            f"STDERR:\n{result.stderr}"
        )

    return result


# ---------------------------------------------------------------------
# 1. LOCATE THE CAPYBARA REPOSITORY
# ---------------------------------------------------------------------

cwd = Path.cwd().resolve()

candidates = [
    cwd,
    cwd / "repository",
    cwd.parent,
    cwd.parent / "repository",
]

# Search a little more broadly for our distinctive RTD config file.
for base in [cwd, cwd.parent]:
    try:
        for p in base.glob("**/.readthedocs.yaml"):
            candidates.append(p.parent)
    except Exception:
        pass

# Remove duplicates while preserving order.
seen = set()
candidates = [
    p for p in candidates
    if not (str(p) in seen or seen.add(str(p)))
]

REPO_ROOT = None

# Prefer an existing Git repository.
for p in candidates:
    if not p.exists():
        continue

    result = subprocess.run(
        ["git", "-C", str(p), "rev-parse", "--show-toplevel"],
        capture_output=True,
        text=True,
    )

    if result.returncode == 0:
        REPO_ROOT = Path(result.stdout.strip())
        break

# If Git was never initialized, locate the source tree by .readthedocs.yaml.
if REPO_ROOT is None:
    for p in candidates:
        if (
            p.exists()
            and (p / ".readthedocs.yaml").exists()
        ):
            REPO_ROOT = p
            break

if REPO_ROOT is None:
    raise RuntimeError(
        "I could not locate the Capybara repository.\n\n"
        "Set REPO_ROOT manually to the folder containing "
        ".readthedocs.yaml and run this cell again."
    )

print("\nRepository selected:")
print(REPO_ROOT)


# ---------------------------------------------------------------------
# 2. SAFETY CHECK
# ---------------------------------------------------------------------

expected_markers = [
    ".readthedocs.yaml",
    "README.md",
]

found = [
    name
    for name in expected_markers
    if (REPO_ROOT / name).exists()
]

print("\nRepository markers found:")
for x in found:
    print("  ✓", x)

if ".readthedocs.yaml" not in found:
    raise RuntimeError(
        "The selected folder does not contain .readthedocs.yaml. "
        "Refusing to initialize/push the wrong directory."
    )


confirm = input(
    "\nType PUSH CAPYBARA to configure and push this repository: "
).strip()

if confirm != "PUSH CAPYBARA":
    raise RuntimeError("Cancelled.")


# ---------------------------------------------------------------------
# 3. INITIALIZE GIT IF NECESSARY
# ---------------------------------------------------------------------

git_check = subprocess.run(
    ["git", "-C", str(REPO_ROOT), "rev-parse", "--git-dir"],
    capture_output=True,
    text=True,
)

if git_check.returncode != 0:
    print("\nInitializing Git repository...")
    run_git(REPO_ROOT, "init")

else:
    print("\n✓ Git repository already initialized.")


# ---------------------------------------------------------------------
# 4. CONFIGURE ORIGIN
# ---------------------------------------------------------------------

remotes = run_git(
    REPO_ROOT,
    "remote",
    check=False,
).stdout.split()

print("\nExisting remotes:", remotes or "(none)")

if "origin" in remotes:

    print("\nUpdating existing origin...")
    run_git(
        REPO_ROOT,
        "remote",
        "set-url",
        "origin",
        GITHUB_URL,
    )

else:

    print("\nNo origin exists — creating it...")
    run_git(
        REPO_ROOT,
        "remote",
        "add",
        "origin",
        GITHUB_URL,
    )


print("\nConfigured remote:")
run_git(
    REPO_ROOT,
    "remote",
    "-v",
)


# ---------------------------------------------------------------------
# 5. STAGE SOURCE
# ---------------------------------------------------------------------

print("\nStaging repository files...")

run_git(
    REPO_ROOT,
    "add",
    "-A",
)


# ---------------------------------------------------------------------
# 6. CREATE INITIAL COMMIT IF NEEDED
# ---------------------------------------------------------------------

head = run_git(
    REPO_ROOT,
    "rev-parse",
    "--verify",
    "HEAD",
    check=False,
)

if head.returncode != 0:

    print("\nNo Git commit exists yet.")

    # Check that Git identity exists.
    username = run_git(
        REPO_ROOT,
        "config",
        "user.name",
        check=False,
    ).stdout.strip()

    email = run_git(
        REPO_ROOT,
        "config",
        "user.email",
        check=False,
    ).stdout.strip()

    if not username:
        print("\nGit user.name is not configured.")
        username = input("Git display name: ").strip()

        run_git(
            REPO_ROOT,
            "config",
            "user.name",
            username,
        )

    if not email:
        print("\nGit user.email is not configured.")
        email = input(
            "Git email address "
            "(or your GitHub no-reply address): "
        ).strip()

        run_git(
            REPO_ROOT,
            "config",
            "user.email",
            email,
        )

    print("\nCreating initial Capybara commit...")

    run_git(
        REPO_ROOT,
        "commit",
        "-m",
        "Reconstruct and expand Capybara Framework documentation",
    )

else:

    print("\n✓ Repository already has commits.")

    # Commit any staged changes, but don't fail if there aren't any.
    status = run_git(
        REPO_ROOT,
        "status",
        "--porcelain",
        check=False,
    ).stdout.strip()

    if status:
        print("\nCommitting current repository changes...")

        run_git(
            REPO_ROOT,
            "commit",
            "-m",
            "Update Capybara Framework release candidate",
        )
    else:
        print("✓ Working tree is already committed.")


# ---------------------------------------------------------------------
# 7. RENAME CURRENT BRANCH TO main
# ---------------------------------------------------------------------

print("\nEnsuring branch is named main...")

run_git(
    REPO_ROOT,
    "branch",
    "-M",
    "main",
)


# ---------------------------------------------------------------------
# 8. SHOW EXACT COMMIT TO BE PUSHED
# ---------------------------------------------------------------------

commit = run_git(
    REPO_ROOT,
    "rev-parse",
    "HEAD",
).stdout.strip()

print("\nCommit to push:")
print(commit)


# ---------------------------------------------------------------------
# 9. PUSH TO YOUR NEW GITHUB REPOSITORY
# ---------------------------------------------------------------------

print("\nPushing to:")
print(GITHUB_URL)

push = subprocess.run(
    [
        "git",
        "-C",
        str(REPO_ROOT),
        "push",
        "-u",
        "origin",
        "main",
    ],
    text=True,
)

if push.returncode != 0:
    raise RuntimeError(
        "\nThe repository configuration is now repaired, "
        "but GitHub rejected or interrupted the push.\n\n"
        "If a Git Credential Manager/browser authentication window "
        "appeared, complete it and run ONLY the push portion again."
    )


# ---------------------------------------------------------------------
# 10. VERIFY THAT GITHUB NOW ADVERTISES refs/heads/main
# ---------------------------------------------------------------------

verify = subprocess.run(
    [
        "git",
        "ls-remote",
        "--heads",
        GITHUB_URL,
        "refs/heads/main",
    ],
    capture_output=True,
    text=True,
)

if verify.returncode != 0:
    raise RuntimeError(
        "GitHub could not be queried after the push:\n"
        + verify.stderr
    )

if not verify.stdout.strip():
    raise RuntimeError(
        "Push completed, but GitHub still does not advertise "
        "refs/heads/main."
    )

remote_commit = verify.stdout.split()[0]

print("\n" + "=" * 72)
print("GITHUB REPOSITORY REPAIRED")
print("=" * 72)

print("\nRepository:")
print(
    "https://github.com/"
    "BrianBowers-NapaCounty/"
    "itam-itsm_integration_framework"
)

print("\nBranch:")
print("main")

print("\nRemote commit:")
print(remote_commit)

print("\n✓ refs/heads/main now exists on GitHub.")
print("=" * 72)


Repository selected:
C:\Users\BBOWERS\Jupyter Notebooks\Capybara_Framework_GitHub_Repository_RC1

Repository markers found:
  ✓ .readthedocs.yaml
  ✓ README.md
